In [ ]:
/**
 * ============================================================================
 *  DNS (DOMAIN NAME SYSTEM) - Complete Simple Notes (Hinglish)
 *  Beginners ke liye - Bilkul simple bhasha mein
 * ============================================================================
 *
 * ============================================================================
 *  INTRO: DNS KYUN ZAROORI HAI?
 * ============================================================================
 *
 * Jab aap kisi service pe jaate ho jaise api.example.com, toh pehla sawaal
 * simple hota hai: "Is NAAM KO KYA NETWORK ADDRESS USE KARNA CHAHIYE?"
 *
 * DNS ka full form: "Domain Name System"
 * - DNS yeh sawaal ka jawaab deta hai
 * - Yeh HUMAN-FRIENDLY NAMES ko un RECORDS mein badalta hai
 *   jinhe clients ko connect hone se PEHLE chahiye
 *
 * DNS BAHUT IMPORTANT hai kyunki:
 *   - Jab aap ek domain ko load balancer pe point karte ho
 *   - Traffic ko regions ke beech move karte ho
 *   - Mail aur security settings publish karte ho
 *   - Toh DNS AKSAR involved hota hai
 *
 * DNS SYSTEM DESIGN mein matter karta hai kyunki yeh:
 *   - HAR NETWORK CONNECTION SE PEHLE baithta hai
 *   - Agar DNS SLOW hai, WRONG hai, BLOCKED hai, ya MISCONFIGURED hai
 *   - Toh application ko request KABHI MIL BHI NAHI SAKTI
 *
 * DNS DESIGN SE CACHED hai:
 *   - Isse yeh FAST aur SCALABLE hai
 *   - Lekin iska matlab hai ki changes TURANT HAR JAGAH NAHI HOTE
 *
 * ============================================================================
 *  1. DNS KYA KARTA HAI (What DNS Does)
 * ============================================================================
 *
 * DNS ek DOMAIN NAME ko ek ya zyada RESOURCE RECORDS se map karta hai
 *
 * RESOURCE RECORD kya hota hai:
 *   - Bas ek TYPED DNS ANSWER hai (typed matlab uska type hota hai)
 *   - Ek record keh sakta hai: "yeh naam is IPv4 address ki taraf point
 *     karta hai"
 *   - Dusra record keh sakta hai: "is domain ka mail is mail server pe jaata hai"
 *
 * ----------------------------------------------------------------------------
 *  EXAMPLES (DNS Records ke):
 * ----------------------------------------------------------------------------
 *   api.example.com.     300  IN  A     203.0.113.10
 *   api.example.com.     300  IN  AAAA  2001:db8::10
 *   www.example.com.     300  IN  CNAME edge.example-cdn.net.
 *   example.com.        3600  IN  MX 10 mail.example.com.
 *
 * ============================================================================
 *  HAR RECORD KE PARTS (Each Record Has:)
 * ============================================================================
 *
 *   1. NAME: Domain name jaise api.example.com
 *
 *   2. TTL: Kitni der tak resolvers answer CACHE kar sakte hain
 *
 *   3. CLASS: Hamesha IN hota hai (Internet ke liye)
 *
 *   4. TYPE: A, AAAA, CNAME, MX, TXT, aur aise hi
 *
 *   5. VALUE: Record ka data
 *
 * ============================================================================
 *  DNS TREE (DNS Ek Ped Jaisa Hai)
 * ============================================================================
 *
 * DNS ek TREE ki tarah organize hota hai:
 *
 *   - ROOT (jar) resolvers ko TOP-LEVEL DOMAINS ki taraf point karta hai
 *     jaise .com
 *   - .com REGISTRY resolvers ko example.com ke NAME SERVERS ki taraf
 *     point karta hai
 *   - Woh NAME SERVERS actual DNS records rakhte hain example.com ke liye
 *
 * Is HANDOFF ko kehte hain DELEGATION (soopadan):
 *   - Simple matlab: "Is hisse ke jawaab ke liye IN name servers se poochho"
 *
 * Diagram flow:
 *   Root (.) -> "Delegates .com" -> TLD (.com) ->
 *   "Delegates example.com" -> Authoritative zone (example.com) ->
 *   "Answers api.example.com" -> Record (api.example.com)
 *
 * ============================================================================
 *  ROOT SERVERS (Sabse Upar Ke Servers)
 * ============================================================================
 *
 * - 13 LOGICAL ROOT SERVER NAMES hain, jinhe A se M tak label kiya gaya hai
 * - Yeh sirf 13 PHYSICAL MACHINES NAHI hain
 * - Har naam DUNIYA BHAR KI BAHUT SAARI MACHINES se serve hota hai
 * - Usually ANYCAST use karke, taaki clients kisi NEARBY INSTANCE tak
 *   pahunch sakein
 *
 * ============================================================================
 *  2. DNS RESOLUTION KAISE KAAM KARTA HAI
 * ============================================================================
 *
 * Zyada tar applications FINAL DNS SERVER se DIRECTLY baat NAHI karte
 *
 * Woh ek local STUB RESOLVER se poochte hain:
 *   - Yeh usually operating system ya runtime ka hissa hota hai
 *   - Stub resolver sawaal ko ek RECURSIVE RESOLVER ko forward karta hai
 *   - Recursive resolver woh LOOKUP KAAM karta hai
 *
 * ============================================================================
 *  COMMON RECURSIVE RESOLVERS (Kaun Hote Hain Yeh)
 * ============================================================================
 *
 *   1. ISP resolvers (aapke internet provider ke)
 *   2. Enterprise resolvers (companies ke)
 *   3. Cloud VPC resolvers (cloud ke)
 *   4. Kubernetes DNS components
 *   5. Public resolvers jaise:
 *      - Cloudflare 1.1.1.1
 *      - Google Public DNS 8.8.8.8
 *
 * ============================================================================
 *  RESOLUTION KA FLOW (Step by Step)
 * ============================================================================
 *
 * Agar recursive resolver ke paas PEHLE SE CACHED ANSWER hai:
 *   - Toh woh TURANT woh return kar deta hai
 *
 * Agar NAHI hai:
 *   - Toh woh DNS TREE follow karta hai jab tak woh SAHI AUTHORITATIVE NAME
 *     SERVER na dhoond le
 *
 * Diagram flow:
 *   Application -> "Resolve name + Address" -> Stub Resolver (OS/library)
 *   -> "Forward query / Cached result" -> Recursive Resolver (ISP/cloud/public)
 *   -> "Follow DNS tree if cache misses" -> Authoritative DNS (Zone owner)
 *   -> "Answer + TTL" -> Resolver -> Client
 *
 * ----------------------------------------------------------------------------
 *  STEP BY STEP (Dheere Dheere):
 * ----------------------------------------------------------------------------
 *
 *   1. Application operating system se poochta hai ki api.example.com
 *      resolve kare
 *
 *   2. Operating system ka STUB RESOLVER ek RECURSIVE RESOLVER se poochta hai
 *
 *   3. Recursive resolver ek ROOT SERVER se poochta hai ki .com kahan milega
 *
 *   4. Root server ek REFERRAL return karta hai
 *      - Iska matlab hai: "Jaa ke .com name servers se poochho"
 *
 *   5. Resolver ek .com TLD SERVER se poochta hai ki example.com kahan milega
 *
 *   6. TLD server example.com ke AUTHORITATIVE NAME SERVERS return karta hai
 *
 *   7. Resolver ek AUTHORITATIVE NAME SERVER se api.example.com ke liye
 *      poochta hai
 *
 *   8. Authoritative server requested RECORDS aur TTL return karta hai
 *
 *   9. Resolver answer CACHE karta hai aur client ko return karta hai
 *
 * Sequence diagram flow:
 *   Application -> Stub Resolver: "Resolve api.example.com"
 *   Stub Resolver -> Recursive Resolver: "Query A/AAAA"
 *   Recursive Resolver -> Root: "Where is .com?"
 *   Root -> Recursive Resolver: ".com NS"
 *   Recursive Resolver -> TLD: "Where is example.com?"
 *   TLD -> Recursive Resolver: "example.com name servers"
 *   Recursive Resolver -> Authoritative DNS: "Resolve api.example.com"
 *   Authoritative DNS -> Recursive Resolver: "A/AAAA answer + TTL"
 *   Recursive Resolver -> Application: "Return"
 *   -> Stub Resolver: "Address result"
 *
 * ============================================================================
 *  CACHING KYUN HOTA HAI (Authoritative Providers Har Lookup NAHI Dekhte)
 * ============================================================================
 *
 * Yeh caching ki wajah se hi AUTHORITATIVE DNS PROVIDERS HAR USER LOOKUP
 * NAHI DEKHTE
 *
 * Woh usually DEKHTE HAIN:
 *   - Recursive resolvers se CACHE MISSES
 *   - HAR REQUEST NAHI (har browser, phone, ya backend service se nahi)
 *
 * ============================================================================
 *  DNS KA TRANSPORT (Kis Port Pe Chalta Hai)
 * ============================================================================
 *
 * TRADITIONAL DNS:
 *   - Normal lookups ke liye UDP port 53 use karta hai
 *   - Yeh TCP port 53 bhi use kar sakta hai, especially jab:
 *     * Response UDP ke liye BAHUT BADA ho
 *     * Ya name servers EK DUSRE se ZONE DATA transfer karte hon
 *
 * ENCRYPTED DNS (Alag Transports Use Karta Hai):
 *   - DoT (DNS over TLS) usually TLS port 853 pe chalta hai
 *   - DoH (DNS over HTTPS) DNS queries ko HTTPS port 443 pe bhejta hai
 *
 * ============================================================================
 *  3. RECURSIVE vs ITERATIVE QUERIES (Do Tarah Ke Sawaal)
 * ============================================================================
 *
 * DNS DO QUERY PATTERNS use karta hai
 * - Naam formal lagte hain, lekin idea simple hai
 *
 * ----------------------------------------------------------------------------
 *  RECURSIVE QUERY:
 * ----------------------------------------------------------------------------
 * - Client resolver se kehta hai: "Mere liye FINAL ANSWER dhoond do"
 * - Resolver POORA KAAM khud karta hai
 *
 * ----------------------------------------------------------------------------
 *  ITERATIVE QUERY:
 * ----------------------------------------------------------------------------
 * - Ek server kehta hai: "Mere paas final answer nahi hai, lekin yeh raha
 *   AGLA SERVER jisse poochna hai"
 * - Requester WAHAN SE AAGE badhta hai
 *
 * ============================================================================
 *  QUERY TYPE COMPARISON TABLE
 * ============================================================================
 *
 *   ┌──────────────┬─────────────────────────────┬────────────────────────────────┐
 *   │ Query Type   │ Who Uses It                 │ Behavior                       │
 *   ├──────────────┼─────────────────────────────┼────────────────────────────────┤
 *   │ Recursive    │ Client to recursive         │ "Find the final answer for me" │
 *   │              │ resolver                    │ (Mere liye final answer dhoond)│
 *   ├──────────────┼─────────────────────────────┼────────────────────────────────┤
 *   │ Iterative    │ Recursive resolver to       │ "Here is the next server to    │
 *   │              │ DNS tree                    │ ask" (Yeh raha agla server)    │
 *   └──────────────┴─────────────────────────────┴────────────────────────────────┘
 *
 * ============================================================================
 *  NORMAL USE MEIN KYA HOTA HAI
 * ============================================================================
 *
 * Normal use mein:
 *   - Aapka laptop ya application apne CONFIGURED RESOLVER ko ek RECURSIVE
 *     QUERY bhejta hai
 *   - Phir woh recursive resolver ROOT, TLD, aur AUTHORITATIVE SERVERS ke
 *     khilaf ITERATIVE QUERIES use karta hai
 *
 * ============================================================================
 *  YEH FARK DEBUGGING MEIN MATTER KARTA HAI
 * ============================================================================
 *
 * Agar yeh ho:
 *   - dig @8.8.8.8 api.example.com  -> KAAM KARTA HAI
 *   - Lekin dig @ns1.example-dns.com api.example.com  -> FAIL KARTA HAI
 *
 * Toh matlab:
 *   - Public resolver bas ek PURANA CACHED ANSWER return kar raha hai
 *   - Jabki AUTHORITATIVE SERVER BROKEN hai
 *
 * ============================================================================
 *  4. DNS SIMULATION (DNS Kaise Kaam Karta Hai - Visual)
 * ============================================================================
 *
 * DNS Resolution:
 *   - Ek recursive resolver ko DNS HIERARCHY walk karte hue dekho
 *   - Aur woh jo seekhta hai use CACHE karta hai
 *
 * Modes: Cold lookups, Warm lookup, Cache hit, Iterative, Recursive
 *
 * Components:
 *   CLIENT (Stub resolver) -> RECURSIVE RESOLVER (Recursive hub) ->
 *   DNS HIERARCHY:
 *     - ROOT (. Root server)
 *     - TLD .COM (gTLD server)
 *     - AUTHORITATIVE (Owner NS)
 *
 * RESOLVER CACHE: Empty cache se shuru hota hai (0 records)
 *
 * ============================================================================
 *  5. DNS COMPONENTS (DNS Ke Hisse - Kaun Kya Karta Hai)
 * ============================================================================
 *
 * Ek DNS lookup KAI SYSTEMS se guzarta hai
 * - Woh AKSAR ALAG TEAMS ya COMPANIES ke paas hote hain
 * - Har ek ka ALAG KAAM hai
 * - Yeh pieces jaanna usse BAHUT AASAAN banata hai ki DNS answer KO
 *   CONTROL karta hai aur KAHAN CHEEZEIN TOOT SAKTI HAIN
 *
 * ============================================================================
 *  DOMAIN REGISTRAR (Domain Kahan Register Hota Hai)
 * ============================================================================
 *
 * - Registrar woh jagah hai jahan ek domain REGISTER hota hai
 *   jaise example.com
 * - Yeh CONTROL karta hai ki domain ke liye KAUNSE NAME SERVERS listed hain
 *   registry mein
 *
 * IMPORTANT - DO ALAG CHEEZEIN HAIN:
 *   - DNS RECORDS CHANGE karna
 *   - NAME SERVER DELEGATION change karna
 *
 * Yeh DONO ALAG OPERATIONS hain:
 *   - Ek A RECORD update karna = aapke DNS ZONE ke andar ek answer change karta hai
 *   - Delegation change karna = un NAME SERVERS ko badalta hai jo POORE DOMAIN
 *     ke zimmedar hain
 *
 * ============================================================================
 *  REGISTRY AND TLD SERVERS
 * ============================================================================
 *
 * - Ek REGISTRY ek TOP-LEVEL DOMAIN operate karta hai
 *   jaise .com, .org, ya koi country-code TLD
 *
 * - TLD SERVERS har record jaise api.example.com STORE NAHI karte
 * - Yeh HANDOFF INFORMATION store karte hain jo resolvers ko batate hain ki
 *   example.com ke liye KAUNSE NAME SERVERS jawaab dete hain
 *
 * ============================================================================
 *  AUTHORITATIVE NAME SERVERS
 * ============================================================================
 *
 * - Authoritative name servers ek ZONE ke OFFICIAL DNS RECORDS rakhte hain
 *   jaise example.com
 *
 * Example:
 *   - Agar aap api.example.com ko ek CLOUD LOAD BALANCER pe chalate ho
 *   - Toh authoritative DNS record US load balancer ki taraf point karega
 *   - Agar aap kisi AUR LOAD BALANCER PE MOVE karte ho
 *   - Toh authoritative record badalta hai
 *   - Jab tak purane answer ke cached TTL expire na ho jaayein, clients purana
 *     address use kar sakte hain
 *
 * ============================================================================
 *  RECURSIVE RESOLVERS (Lookup Ka Kaam Karta Hai)
 * ============================================================================
 *
 * - Recursive resolvers CLIENTS ke liye LOOKUP KAAM karte hain aur RESULTS
 *   CACHE karte hain
 * - Yeh RELIABILITY ke liye BAHUT matter karte hain
 * - Ek SLOW ya BROKEN recursive resolver HEALTHY SERVICES ko UNAVAILABLE
 *   dikha sakta hai
 *
 * ============================================================================
 *  STUB RESOLVERS (Client Side Ka Chhota Hissa)
 * ============================================================================
 *
 * - Stub resolver woh CHHOTA CLIENT-SIDE DNS COMPONENT hai:
 *   - Operating system mein
 *   - Runtime mein
 *   - Library mein
 * - Yeh usually DNS questions ko ek RECURSIVE RESOLVER ko forward karta hai
 *   jo configure hota hai:
 *     * DHCP se
 *     * VPN settings se
 *     * Container runtime se
 *     * /etc/resolv.conf se
 *     * Platform settings se
 *
 * IMPORTANT - Stub resolver behaviour ALAG ALAG hota hai:
 *   - Kuch answers CACHE karte hain
 *   - Kuch NAHI karte
 *   - Kuch application runtimes EXPECTED SE ZYADA DER tak DNS cache karte hain
 *   - Java, browsers, mobile SDKs, service meshes, aur libc implementations
 *     SAB ALAG behave kar sakte hain
 *
 * ============================================================================
 *  6. DNS RECORD TYPES (DNS Record Ke Types)
 * ============================================================================
 *
 * DNS ALAG ALAG TARAH KE records store karta hai
 * - Yeh woh hain jo engineers PRODUCTION MEIN AKSAR DEKHTE hain
 *
 *   ┌──────────────┬─────────────────────────────────┬────────────────────────────────────┐
 *   │ Record       │ Purpose                         │ Example                            │
 *   ├──────────────┼─────────────────────────────────┼────────────────────────────────────┤
 *   │ A            │ Ek naam ko IPv4 address se map  │ api.example.com -> 203.0.113.10    │
 *   │              │ karta hai                       │                                    │
 *   ├──────────────┼─────────────────────────────────┼────────────────────────────────────┤
 *   │ AAAA         │ Ek naam ko IPv6 address se map  │ api.example.com -> 2001:db8::10    │
 *   │              │ karta hai                       │                                    │
 *   ├──────────────┼─────────────────────────────────┼────────────────────────────────────┤
 *   │ CNAME        │ Ek naam ko dusre naam ki taraf  │ www.example.com -> edge.provider.  │
 *   │              │ point karta hai                 │ net                                │
 *   ├──────────────┼─────────────────────────────────┼────────────────────────────────────┤
 *   │ MX           │ Ek domain ke liye mail routing  │ example.com -> mail.example.com    │
 *   ├──────────────┼─────────────────────────────────┼────────────────────────────────────┤
 *   │ TXT          │ Verification aur policy ke liye │ SPF, DKIM, ownership verification  │
 *   │              │ text data                       │                                    │
 *   ├──────────────┼─────────────────────────────────┼────────────────────────────────────┤
 *   │ NS           │ Ek zone ko name servers pe      │ example.com -> ns1.provider.net    │
 *   │              │ delegate karta hai              │                                    │
 *   ├──────────────┼─────────────────────────────────┼────────────────────────────────────┤
 *   │ SOA          │ Zone ownership aur timing       │ Serial, refresh, retry, negative   │
 *   │              │ metadata                        │ cache TTL                          │
 *   ├──────────────┼─────────────────────────────────┼────────────────────────────────────┤
 *   │ CAA          │ Restrict karta hai ki kaunse    │ letsencrypt.org                    │
 *   │              │ CAs certificates issue kar sakte│                                    │
 *   │              │ hain                            │                                    │
 *   ├──────────────┼─────────────────────────────────┼────────────────────────────────────┤
 *   │ SRV          │ Port aur priority ke saath      │ Some internal and legacy service   │
 *   │              │ service location                │ discovery                          │
 *   ├──────────────┼─────────────────────────────────┼────────────────────────────────────┤
 *   │ HTTPS /      │ Modern service connection setup │ HTTP/3, alternative endpoints,     │
 *   │ SVCB         │ ke liye hints                   │ ECH-related deployment             │
 *   └──────────────┴─────────────────────────────────┴────────────────────────────────────┘
 *
 * ============================================================================
 *  DO PRACTICAL NOTES (2 Zaroori Baatein)
 * ============================================================================
 *
 * ----------------------------------------------------------------------------
 *  NOTE 1: CNAME KYA KARTA HAI
 * ----------------------------------------------------------------------------
 *   - Ek CNAME DOOSRE NAAM ki taraf point karta hai
 *   - DIRECTLY IP ADDRESS KI TARAF NAHI
 *   - Ek CNAME dekhne ke BAAD bhi resolver ko abhi bhi TARGET NAME resolve
 *     karna padta hai
 *
 * ----------------------------------------------------------------------------
 *  NOTE 2: CNAME DOMAIN ROOT PE NAHI HO SAKTA
 * ----------------------------------------------------------------------------
 *   - Standard DNS ek CNAME ko DOMAIN KE ROOT pe allow NAHI karta
 *     (jaise example.com)
 *   - Kyunki wahan required records jaise SOA aur NS PEHLE SE exist karte hain
 *   - Isliye bahut saare providers offer karte hain:
 *     * ALIAS
 *     * ANAME
 *     * CNAME flattening
 *   - Yeh PROVIDER-SPECIFIC tarike hain domain root pe CNAME-LIKE behaviour
 *     paane ke liye
 *
 * ============================================================================
 *  7. TTL AND CACHING (Cache Kitni Der Ke Liye)
 * ============================================================================
 *
 * - DNS caching TTL (Time To Live) se CONTROL hota hai
 * - TTL resolvers ko batata hai ki woh ek answer KITNI DER tak REUSE kar sakte
 *   hain PHIR SE POOCHNE SE PEHLE
 *
 * Example:
 *   api.example.com.  60  IN  A  203.0.113.10
 *   -> Koi bhi resolver jo yeh answer receive karta hai, ise 60 SECONDS tak
 *      cache kar sakta hai
 *
 * ============================================================================
 *  TTL EK TRADEOFF HAI (Fayda Aur Nuksan)
 * ============================================================================
 *
 *   ┌─────────────────────┬──────────────────────────┬─────────────────────────────────────┐
 *   │ TTL                 │ Behavior                 │ Tradeoff                            │
 *   ├─────────────────────┼──────────────────────────┼─────────────────────────────────────┤
 *   │ 30-60 seconds       │ Faster changes aur       │ More DNS query volume               │
 *   │                     │ failover                 │ (zyada DNS queries)                 │
 *   ├─────────────────────┼──────────────────────────┼─────────────────────────────────────┤
 *   │ 300-3600 seconds    │ Many services ke liye    │ Changes are not instant             │
 *   │                     │ good default             │ (changes turant nahi hote)          │
 *   ├─────────────────────┼──────────────────────────┼─────────────────────────────────────┤
 *   │ 3600+ seconds       │ Cache-friendly aur       │ Poor fit for active traffic shifts  │
 *   │                     │ stable                   │ (active changes ke liye bura)       │
 *   └─────────────────────┴──────────────────────────┴─────────────────────────────────────┘
 *
 * ============================================================================
 *  IMPORTANT - LOW TTL KA MATLAB INSTANT CHANGE NAHI HAI
 * ============================================================================
 *
 *   - EXISTING CACHES old answers ko TAB TAK RAKH SAKTE HAIN jab tak expire na
 *     ho jaayein
 *   - Kuch resolvers apne KHUD KE MINIMUM TTLs enforce karte hain
 *   - Kuch CLIENTS apne answers KHUD cache karte hain
 *   - Kuch applications EK BAAR STARTUP pe name resolve karte hain
 *     - Aur PHIR KABHI REFRESH NAHI karte jab tak process restart na ho
 *
 * ============================================================================
 *  NEGATIVE CACHING (Failures Bhi Cache Hote Hain)
 * ============================================================================
 *
 * DNS FAILURES bhi CACHE karta hai
 * - Agar ek resolver NXDOMAIN receive karta hai
 *   (matlab "yeh naam exist NAHI karta")
 * - Toh woh us failure ko THODI DER ke liye CACHE kar sakta hai
 *
 * YEH ROLLOUTS KE DAURAAN MATTER KARTA HAI:
 *   - Agar clients new-api.example.com ko look up karte hain
 *   - AAPKE RECORD BANAANE SE PEHLE
 *   - Toh woh failure CACHE kar sakte hain
 *   - Aur KUCH DER TAK FAIL HOTE RAH SAKTE HAIN
 *   - CHAHE RECORD AB EXIST KARTA HO
 *
 * ============================================================================
 *  CNAME CHAINS (CNAME Ki Kadi)
 * ============================================================================
 *
 * CNAME chains EXTRA LOOKUP STEPS add karte hain:
 *
 *   www.example.com.       300  IN  CNAME  example.cdn.net.
 *   example.cdn.net.        60  IN  CNAME  edge.provider.net.
 *   edge.provider.net.      60  IN  A      203.0.113.20
 *
 * IMPORTANT:
 *   - CHHOTI CHAINS NORMAL hain
 *   - LAMBI CHAINS:
 *     * LATENCY add karti hain
 *     * TOOTNE KE ZYADA JAGAH banati hain
 *     * Migrations ko samajhna MUSHKIL banati hain
 *
 * ============================================================================
 *  8. DNS AND TRAFFIC STEERING (Traffic Ko Modna)
 * ============================================================================
 *
 * DNS SAME HOSTNAME ke liye ALAG ALAG ANSWERS return karke TRAFFIC STEER kar
 * sakta hai
 *
 * COMMON POLICIES (Aam Rules):
 *   1. WEIGHTED RECORDS (vazandar records)
 *   2. LATENCY-BASED ANSWERS (delay ke hisaab se)
 *   3. HEALTH-CHECK-BASED FAILOVER (swasthya check se)
 *   4. REGIONAL ROUTING (region ke hisaab se)
 *   5. BLUE-GREEN OR CANARY MIGRATION
 *
 * ============================================================================
 *  DNS STEERING - EK BROAD TOOL HAI (Limitations)
 * ============================================================================
 *
 * DNS steering USEFUL hai, lekin yeh ek BROAD TOOL hai:
 *   - Yeh tab hota hai jab NAME RESOLVE hota hai
 *   - TAB NAHI jab HAR REQUEST handle hoti hai
 *
 * Ek resolver EK ANSWER CACHE kar sakta hai aur usse BAHUT SAARE CLIENTS ke
 * liye REUSE kar sakta hai
 *
 * DNS YEH BHI NAHI JAANTA:
 *   - Current CPU load
 *   - Open connection count
 *   - Queue depth
 *   - User identity
 *   - HTTP path
 *
 * BEST PRACTICE:
 *   - DNS use karo GLOBAL ya REGIONAL ENTRY POINT choose karne ke liye
 *   - Load balancers, gateways, service meshes, ya application routing use karo
 *     UN decisions ke liye jo PER REQUEST hone chahiye
 *
 * Example diagram:
 *   Client -> Recursive Resolver: "Resolve api.example.com"
 *   Recursive Resolver -> Authoritative DNS: "Cache miss" (geo policy)
 *   Authoritative DNS -> Recursive Resolver: "Returns regional answer"
 *   Recursive Resolver -> Client: "Address"
 *   US clients -> US Load Balancer: 203.0.113.10
 *   EU clients -> EU Load Balancer: 198.51.100.10
 *
 * DNS REGION choose karne mein madad karta hai
 * - Us address ke PEECHE wala LOAD BALANCER decide karta hai ki request ko
 *   ACTUALLY KAUNSA SERVER handle karega
 *
 * ============================================================================
 *  9. DNS SECURITY (DNS Ki Suraksha)
 * ============================================================================
 *
 * DNS ke paas KAI SECURITY CONCERNS hain
 * - SAHI PROTECTION ispe depend karti hai ki aap KYA PREVENT karna chahte ho
 *
 * ============================================================================
 *  DNSSEC
 * ============================================================================
 *
 * DNSSEC kya karta hai:
 *   - DNS records ko SIGN karta hai
 *   - Taaki resolvers CHECK kar sakein ki answers DOMAIN OWNER se aaye hain
 *   - Aur unhe RAASTE MEIN CHANGE NAHI kiya gaya
 *
 * DNSSEC KYA NAHI KARTA:
 *   - Yeh DNS queries ENCRYPT NAHI karta
 *   - Network pe dekhne wala koi bhi ABHI BHI DEKH SAKTA HAI ki
 *     KAUNSA NAME query ho raha hai
 *   - Jab tak koi aur PRIVACY TOOL use na kiya jaaye
 *
 * DNSSEC OPERATIONALLY KYA ADD KARTA HAI:
 *   - KEY MANAGEMENT add karta hai
 *   - BROKEN SIGNATURES, EXPIRED KEYS, ya BAD DS RECORDS
 *     ek domain ko DNS VALIDATION fail kara sakte hain
 *   - CHAHE application KHUD HEALTHY ho
 *
 * ============================================================================
 *  DoT AND DoH (Encrypted DNS)
 * ============================================================================
 *
 *   - DNS over TLS (DoT)
 *   - DNS over HTTPS (DoH)
 *   - Yeh DNS traffic ko CLIENT aur RECURSIVE RESOLVER ke beech ENCRYPT karte hain
 *
 * Yeh KYA PROTECT KARTE HAIN:
 *   - Client aur resolver ke beech ke PATH ko SNOOPING ya TAMPERING se
 *
 * Yeh KYA PROTECT NAHI KARTE:
 *   - Yeh queries ko RECURSIVE RESOLVER KHUD se HIDE NAHI karte
 *   - Yeh DNSSEC ko REPLACE NAHI karte
 *
 * ============================================================================
 *  CACHE POISONING AND SPOOFING
 * ============================================================================
 *
 * DNS CACHE POISONING kya karta hai:
 *   - Ek resolver ko TRICK karta hai ki woh ek FAKE ANSWER CACHE kare
 *
 * MODERN RESOLVERS ke paas BAHUT SAARE DEFENSES hain:
 *   1. Random IDs
 *   2. Random source ports
 *   3. Strict checks on which servers are allowed to answer
 *   4. DNSSEC validation
 *   5. Hardened resolver behaviour
 *
 * ============================================================================
 *  APPLICATION TEAMS KE LIYE MAIN LESSON (Simple Hai):
 * ============================================================================
 *
 *   1. REPUTABLE DNS PROVIDERS use karo
 *
 *   2. DNSSEC enable karo jab aap ise CORRECTLY OPERATE kar sako
 *
 *   3. REGISTRAR ACCESS protect karo
 *
 *   4. DOMAIN aur CERTIFICATE CHANGES monitor karo
 *
 * ============================================================================
 *  10. PRIVATE DNS AND SERVICE DISCOVERY
 * ============================================================================
 *
 * DNS sirf PUBLIC INTERNET ke liye NAHI hai
 *
 * ============================================================================
 *  PRIVATE DNS KAHAN COMMON HAI
 * ============================================================================
 *
 *   1. Cloud VPCs
 *   2. Kubernetes clusters
 *   3. Corporate networks
 *   4. VPN-connected environments
 *   5. Service meshes aur internal platforms
 *
 * ============================================================================
 *  EXAMPLES (Private DNS Records)
 * ============================================================================
 *
 *   payments.production.svc.cluster.local.  30  IN  A  10.20.4.15
 *   db.internal.example.com.                60  IN  A  10.0.12.8
 *
 * ============================================================================
 *  PRIVATE DNS APNI FAILURE MODES BHI LATA HAI
 * ============================================================================
 *
 *   1. SPLIT-HORIZON RECORDS network ke andar aur bahar ALAG answers return
 *      karte hain
 *
 *   2. VPN CLIENTS galat resolver use kar sakte hain
 *
 *   3. SEARCH DOMAINS aur ndots settings SURPRISING QUERY VOLUME create kar
 *      sakte hain
 *
 *   4. INTERNAL NAMES PUBLIC RESOLVERS pe LEAK ho sakte hain agar resolver
 *      configuration galat hai
 *
 *   5. SHORT TTLs traffic spikes ke dauraan CLUSTER DNS ko OVERLOAD kar sakte
 *      hain
 *
 * ============================================================================
 *  SPLIT-HORIZON DNS KYA HAI
 * ============================================================================
 *
 * Simple matlab:
 *   - EK HI NAAM private network ke andar EK answer return kar sakta hai
 *   - Aur uske bahar DOOSRA answer
 *
 * ============================================================================
 *  ndots KYA HAI
 * ============================================================================
 *
 * ndots ek RESOLVER SETTING hai:
 *   - Yeh control karta hai ki ek naam KOI COMPLETE hai ya NAHI
 *   - Jab SEARCH DOMAINS append kiye jaate hain
 *   - Ek SURPRISING ndots value EK LOOKUP ko KAI LOOKUPS mein badal sakta hai
 *
 * ============================================================================
 *  KUBERNETES MEIN DNS
 * ============================================================================
 *
 * Kubernetes mein:
 *   - DNS ek COMMON SERVICE DISCOVERY INTERFACE hai
 *   - Lekin HIGH-TRAFFIC SYSTEMS ko abhi bhi samajhna chahiye:
 *     * Caching
 *     * Connection reuse
 *     * Service mesh behaviour
 *
 * IMPORTANT:
 *   - HAR REQUEST pe naam resolve karna usually EK BURI DESIGN hai
 *
 * ============================================================================
 *  11. DNS FAILURE MODES (DNS Kab Fail Hota Hai)
 * ============================================================================
 *
 * DNS failures AKSAR APPLICATION FAILURES jaise dikhte hain
 * - Jab tak aap RESOLUTION ko EXPLICITLY CHECK na karo
 *
 * ============================================================================
 *  TROUBLESHOOTING TABLE (Problem Aur Karan)
 * ============================================================================
 *
 *   ┌──────────────────────────────────┬─────────────────────────────────────────────┐
 *   │ Symptom (Problem)                │ Likely Cause (Sambhavit Karan)              │
 *   ├──────────────────────────────────┼─────────────────────────────────────────────┤
 *   │ NXDOMAIN                         │ Name exist nahi karta, wrong zone, ya       │
 *   │                                  │ cached failure                              │
 *   ├──────────────────────────────────┼─────────────────────────────────────────────┤
 *   │ SERVFAIL                         │ DNSSEC failure, authoritative outage, ya    │
 *   │                                  │ resolver issue                              │
 *   ├──────────────────────────────────┼─────────────────────────────────────────────┤
 *   │ Slow first request               │ Slow DNS lookup, long CNAME chain, ya       │
 *   │                                  │ resolver problem                            │
 *   ├──────────────────────────────────┼─────────────────────────────────────────────┤
 *   │ Some users reach old endpoint    │ Old answers abhi bhi kahin cache hain       │
 *   ├──────────────────────────────────┼─────────────────────────────────────────────┤
 *   │ Works on VPN but not outside     │ Private DNS ya split-horizon DNS dependency │
 *   ├──────────────────────────────────┼─────────────────────────────────────────────┤
 *   │ Works with one resolver but not  │ Cache, DNSSEC, propagation, ya resolver     │
 *   │ another                          │ policy difference                           │
 *   ├──────────────────────────────────┼─────────────────────────────────────────────┤
 *   │ Browser works, backend fails     │ Different resolver path ya application DNS  │
 *   │                                  │ cache                                       │
 *   └──────────────────────────────────┴─────────────────────────────────────────────┘
 *
 * ============================================================================
 *  USEFUL DEBUGGING COMMANDS (Kaise Check Karein)
 * ============================================================================
 *
 *   dig api.example.com A
 *   dig api.example.com AAAA
 *   dig +trace api.example.com
 *   dig @1.1.1.1 api.example.com
 *   dig @8.8.8.8 api.example.com
 *   dig NS example.com
 *   dig SOA example.com
 *
 * ============================================================================
 *  PRODUCTION DNS DEBUG KARTE WAQT - IN ALAG ALAG JAGAHON SE COMPARE KARO:
 * ============================================================================
 *
 *   1. Public resolver vs corporate resolver
 *
 *   2. Inside VPC vs outside VPC
 *
 *   3. Authoritative answer vs cached recursive answer
 *
 *   4. A aur AAAA records
 *
 *   5. DNSSEC-validating vs non-validating resolver
 *
 *   6. Application runtime behaviour vs command-line resolver behaviour
 *
 * ============================================================================
 *  12. DNS IN SYSTEM DESIGN (System Design Mein DNS)
 * ============================================================================
 *
 * DNS deliberately use karo (soch samajh ke)
 * - Ise edit karna SIMPLE hai
 * - Lekin GALTIYAN ko THEEK karna SAMAY LETA hai
 * - Kyunki PURANE ANSWERS CACHED ho sakte hain
 *
 * ============================================================================
 *  USE STABLE NAMES (Stable Naam Use Karo)
 * ============================================================================
 *
 * Applications ko HARD-CODED IP ADDRESSES ke bajaye STABLE NAMES pe depend
 * karna chahiye
 *
 * ----------------------------------------------------------------------------
 *  GOOD (Achha Example):
 * ----------------------------------------------------------------------------
 *   DATABASE_HOST=db.internal.example.com
 *   MODEL_API_URL=https://models.example.com
 *
 * ----------------------------------------------------------------------------
 *  BAD (Bura Example):
 * ----------------------------------------------------------------------------
 *   DATABASE_HOST=10.0.12.8
 *   MODEL_API_URL=https://203.0.113.40
 *
 * NAMES se aap YEH SAB KAR SAKTE HO BINA application configuration badle:
 *   1. Services move karna
 *   2. Infrastructure replace karna
 *   3. Providers change karna
 *   4. Certificates issue karna
 *
 * ============================================================================
 *  PLAN TTLs BEFORE MIGRATIONS (Migration Se Pehle TTLs Plan Karo)
 * ============================================================================
 *
 * PLANNED MIGRATION se PEHLE:
 *   1. TTLs ko EXPIRE hone ka WAIT karo
 *   3. Phir records CHANGE karo
 *   4. Migration STABLE hone ke baad, TTLs PHIR SE LOWER karo
 *   2. Purane caches RAISE karo
 *      agar record ko AKSAR change NAHI karna padta
 *
 * IMPORTANT WARNING:
 *   - WOHI MOMENT mein TTL LOWER MAT KARO jab aapko CHANGE chahiye
 *   - Jo resolvers PEHLE SE old LONGER TTL CACHE kar chuke hain
 *   - Woh tab tak PURANA ANSWER use kar sakte hain jab tak woh EXPIRE NA HO
 *
 * ============================================================================
 *  SEPARATE DNS FROM REQUEST ROUTING (DNS Ko Alag Rakho)
 * ============================================================================
 *
 * DNS ek ACHHI JAGAH hai REGION ya ENTRY POINT choose karne ke liye
 * - Yeh ACHHI JAGAH NAHI hai HAR REQUEST ke liye EXACT BACKEND choose karne ke
 *   liye
 *
 * Example - Ek AI inference platform yeh use kar sakta hai:
 *   1. DNS to route api.example.com to the nearest healthy edge region
 *      (DNS se sabse kareeb healthy edge region pe route karna)
 *
 *   2. A load balancer to distribute connections inside the region
 *      (Load balancer se region ke andar connections baantna)
 *
 *   3. An API gateway to authenticate and rate limit
 *      (API gateway se authenticate aur rate limit karna)
 *
 *   4. An application router to choose a model pool
 *      (Application router se model pool choose karna)
 *
 *   5. Service discovery to find healthy inference workers
 *      (Service discovery se healthy inference workers dhoondhna)
 *
 * ============================================================================
 *  HAR LAYER KA ALAG KAAM HAI
 * ============================================================================
 *
 *   - DNS client ko SAHI FRONT DOOR tak pahunchata hai
 *   - Us door ke PEECHE wali layers REQUEST-LEVEL DECISIONS handle karti hain
 *
 * ============================================================================
 *  SUMMARY (Saransh)
 * ============================================================================
 *
 * 1. DNS ek DISTRIBUTED, CACHED system hai NAMES LOOK UP karne ke liye
 *    - Yeh names ko TYPED RECORDS se map karta hai
 *    - SIRF IP ADDRESSES SE NAHI
 *
 * 2. Clients usually AUTHORITATIVE SERVERS se DIRECTLY NAHI poochte
 *    - Woh RECURSIVE RESOLVERS se poochte hain
 *    - ROOT aur TLD servers resolvers ko AGLE JAGAH ki taraf point karte hain
 *    - AUTHORITATIVE SERVERS apni ZONES ke liye FINAL ANSWERS dete hain
 *
 * 3. TTLs control karte hain ki answers KITNI DER CACHE RAHEIN
 *    - FAILED LOOKUPS bhi included hain
 *    - Isliye DNS changes KABHI TRULY INSTANT NAHI hote
 *    - DNS traffic STEER kar sakta hai, lekin sirf BROADLY
 *    - Aur sirf tab jab ek NAME RESOLVE hota hai
 *
 * 4. SECURITY aur PRIVACY EASY hai MIX UP karne ke liye:
 *    - DNSSEC DNS answers GENUINE hone mein madad karta hai
 *    - Lekin yeh queries ENCRYPT NAHI karta
 *    - DoH aur DoT client aur resolver ke beech traffic ENCRYPT karte hain
 *    - Lekin yeh DNSSEC ko REPLACE NAHI karte
 *
 * 5. PRIVATE DNS cloud aur Kubernetes systems mein ESSENTIAL hai
 *    - Aur yeh apni FAILURE MODES bhi lata hai
 *
 * 6. IMPORTANT - DNS KA DICHOTOMY:
 *    - Jab DNS KAAM karta hai, KOI ISE SOCHTA BHI NAHI
 *    - Jab DNS TOOTTA hai, symptoms aise dikhte hain:
 *      * Slow first requests
 *      * Failed migrations
 *      * Broken certificates
 *      * Intermittent outages
 *      * Unreachable services
 *
 * ============================================================================
 **/